# 01 — SAT demo (CPU, 1 min)

Corre el caso chico (triángulo k4) con Glucose3. Imprime UNA línea JSON.
Pegá esa línea en el chat de Grafito para importar y verificar en local.
No necesita GPU/TPU.

In [ ]:
import json, time, subprocess, sys
from concurrent.futures import ThreadPoolExecutor, TimeoutError
subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "python-sat"])
from pysat.solvers import Glucose3

TIMEOUT_S = 30
CASES = []
CASES.append(("adec367d0c2f9799", "p cnf 12 25\n1 2 3 4 0\n-1 -2 0\n-1 -3 0\n-1 -4 0\n-2 -3 0\n-2 -4 0\n-3 -4 0\n5 6 7 8 0\n-5 -6 0\n-5 -7 0\n-5 -8 0\n-6 -7 0\n-6 -8 0\n-7 -8 0\n9 10 11 12 0\n-9 -10 0\n-9 -11 0\n-9 -12 0\n-10 -11 0\n-10 -12 0\n-11 -12 0\n-1 -5 0\n-2 -6 0\n-3 -7 0\n-4 -8 0\n"))

def parse_dimacs(text):
    clauses = []
    for line in text.splitlines():
        line = line.strip()
        if not line or line.startswith("c") or line.startswith("p"):
            continue
        lits = [int(x) for x in line.split()]
        if lits and lits[-1] == 0:
            lits.pop()
        if lits:
            clauses.append(lits)
    return clauses

def solve_one(clauses):
    with Glucose3() as solver:
        for cl in clauses:
            solver.add_clause(cl)
        sat = solver.solve()
        return sat, solver.get_model() if sat else None

out = []
pool = ThreadPoolExecutor(max_workers=1)
for name, text in CASES:
    clauses = parse_dimacs(text)
    t0 = time.time()
    fut = pool.submit(solve_one, clauses)
    try:
        sat, model = fut.result(timeout=TIMEOUT_S)
        status = "SAT" if sat else "UNSAT"
    except TimeoutError:
        sat, model, status = None, None, "TIMEOUT"
    out.append({"name": name, "status": status, "model": model, "time_s": round(time.time() - t0, 3)})
print(json.dumps({"results": out}))
